### 1.1.10. Sets of Ordered Pairs

A **set of ordered pairs** (a 2-D set) is a finite collection of 2-tuples `(i, j)` in which
order matters: `(i, j)` and `(j, i)` are different elements unless `i == j`.
Two-dimensional sets are the standard way to model binary relationships in AMPL —
origin-destination links in transportation networks, arc sets in graph models,
allowed product-machine assignments in scheduling, and similar structures.

This section provides the full treatment of 2-D sets, covering:
- The `(i, j)` notation and what makes a set 2-dimensional
- Declaring a 2-D set with `dimen 2` or `within ORIG cross DEST`
- Full Cartesian products vs. sparse subsets
- Supplying 2-D set data in `.dat` files
- Indexing over 2-D sets in constraints and sums
- Parameters indexed over 2-D sets
- The `setof` operator for constructing 2-D sets
- Membership test for pairs
- Projections (extracting the first or second component)
- Assigning 2-D sets in amplpy

In [ ]:
# --- Formal grammar: 2-D set declarations ---

# Form 1: explicit dimension only
set NAME dimen 2;

# Form 2: membership constraint using cross operator (preferred)
set NAME within SET_A cross SET_B;

# Form 3: membership constraint using comma-indexing syntax (equivalent to Form 2)
set NAME within {SET_A, SET_B};

# Form 4: full Cartesian product (symbolic definition)
set NAME = SET_A cross SET_B;

# Form 5: symbolic definition with inline filter
set NAME = {i in SET_A, j in SET_B: CONDITION};

# Form 6: setof operator — derive from another set
set NAME = setof {INDEXING_EXPR} (COMPONENT_EXPR);

# --- Indexing over a 2-D set (in constraints, sums, parameters) ---
# When iterating over a 2-D set, dummy indices must be bound as a pair:
#   {(i,j) in NAME}   -- introduces both i and j
#   {(i,j) in NAME}   -- where i is already defined: a row-slice (all j for that i)
#   {(i,j) in NAME}   -- where j is already defined: a column-slice (all i for that j)

#### What Is an Ordered Pair in AMPL?

An **ordered pair** is a 2-tuple written as `(i, j)` — two objects separated by a comma
and surrounded by parentheses. The objects may be strings or numbers.
Order is significant: `("PITT", "STL")` and `("STL", "PITT")` are distinct elements.
An object may appear in both positions, as in `("PITT", "PITT")`.

A **set of ordered pairs** is a finite collection of distinct 2-tuples:

```ampl
{("PITT","STL"), ("PITT","FRE"), ("CLEV","FRE")}
{(1,1), (1,2), (1,3), (2,1), (2,2), (2,3)}
```

The **dimension** of a set is the length of its tuples. A set of single elements has
dimension 1 (the default); a set of ordered pairs has dimension 2. All members of a
set must have the same dimension.

In practice, pairs are rarely enumerated literally in a model. They are either derived
from the full Cartesian product of two simpler sets, or supplied as sparse data
representing only the valid combinations.

**Two notations for the Cartesian product.** AMPL provides two syntactically equivalent
ways to denote the set of all pairs from sets `A` and `B`:

| Form | Syntax | Typical use |
|---|---|---|
| `cross` operator | `A cross B` | Named set declaration, right-hand side of `=` |
| Comma-indexing | `{A, B}` or `{i in A, j in B}` | Inline indexing expressions |

Both forms produce the same set of $|A| \times |B|$ pairs.

#### Declaring a 2-D Set: `dimen` vs. `within`

There are two ways to tell AMPL that a set contains pairs.

**`dimen 2`** tells AMPL the dimension but provides no information about which base sets
the components come from. AMPL will accept any pair from the data; no membership
validation against component sets is performed.

```ampl
set LINKS dimen 2;   # valid — but AMPL cannot verify that components are in ORIG or DEST
```

**`within SET_A cross SET_B`** (or equivalently `within {SET_A, SET_B}`) both declares
the dimension as 2 *and* constrains every data-supplied pair `(i, j)` to satisfy
`i in SET_A` and `j in SET_B`. AMPL enforces this at data-load time, producing an
informative error rather than a silent wrong answer later.

```ampl
set LINKS within ORIG cross DEST;   # preferred: dimension inferred as 2,
                                    # and component membership validated
```

The two forms `within ORIG cross DEST` and `within {ORIG, DEST}` are completely
equivalent. The `cross`-operator form is more explicit; the brace form matches the
inline indexing syntax.

| Declaration | Dimension | Component validation |
|---|---|---|
| `set LINKS dimen 2;` | 2 | None |
| `set LINKS within ORIG cross DEST;` | 2 (inferred) | Yes — each `(i,j)` must have `i in ORIG`, `j in DEST` |
| `set LINKS within {ORIG, DEST};` | 2 (inferred) | Yes — identical to the line above |
| `set LINKS = ORIG cross DEST;` | 2 | N/A — set is fully defined by the expression |

#### Full Cartesian Product vs. Sparse Subset

**Full Cartesian product.** When every pair from `ORIG cross DEST` is valid,
define the set with `=`:

```ampl
set LINKS = ORIG cross DEST;     # all |ORIG| * |DEST| pairs
```

This is appropriate when the model is dense — every origin can ship to every destination.
Parameters and variables indexed over `LINKS` will have one entry per pair.

**Sparse subset.** In realistic network or scheduling models only *some* pairs are valid.
Declare a membership constraint and supply only the valid pairs in the data:

```ampl
set LINKS within ORIG cross DEST;   # sparse: only the data-supplied pairs are members
```

With a sparse set, parameters and variables indexed over `LINKS` are defined only for
the declared pairs. Attempting to reference `cost["GARY", "FRA"]` when
`("GARY", "FRA")` is not in `LINKS` produces an error.

The distinction is illustrated by the difference between the full and the sparse
transportation model tables:

```
Full product (ORIG cross DEST):    Sparse subset (LINKS):
     FRA  DET  LAN  WIN  STL  FRE  LAF       FRA  DET  LAN  WIN  STL  FRE  LAF
GARY  x    x    x    x    x    x    x    GARY       x    x         x         x
CLEV  x    x    x    x    x    x    x    CLEV  x    x    x    x    x         x
PITT  x    x    x    x    x    x    x    PITT  x              x    x    x
```

#### Example 1 — Full Cartesian Product: Dense Transportation Model

This minimal model names the full Cartesian product as `LINKS` and indexes
both parameters and variables over it. The indexing expression `{(i,j) in LINKS}`
introduces both dummy indices simultaneously.

In [ ]:
# Transportation model with full Cartesian product (transp2.mod pattern)
set ORIG;                                    # origin cities
set DEST;                                    # destination cities
set LINKS = ORIG cross DEST;                 # all (origin, destination) pairs

param supply {ORIG} >= 0;                    # supply available at each origin
param demand {DEST} >= 0;                    # demand required at each destination
param cost   {LINKS} >= 0;                   # cost per unit shipped on each link

check: sum {i in ORIG} supply[i] = sum {j in DEST} demand[j];

var Trans {LINKS} >= 0;                      # units shipped on each link

minimize Total_Cost:
    sum {(i,j) in LINKS} cost[i,j] * Trans[i,j];

subject to Supply {i in ORIG}:
    sum {j in DEST} Trans[i,j] = supply[i];  # j iterates freely over all DEST

subject to Demand {j in DEST}:
    sum {i in ORIG} Trans[i,j] = demand[j];  # i iterates freely over all ORIG

When `LINKS = ORIG cross DEST`, every `(i, j)` combination is defined, so the
Supply and Demand constraints may iterate freely over `DEST` and `ORIG` respectively
without a membership guard. The supply constraint `sum {j in DEST} Trans[i,j]`
is valid because `Trans[i,j]` exists for every `j`.

The equivalent declaration `set LINKS = {ORIG, DEST};` is identical in meaning
to `set LINKS = ORIG cross DEST;`.

#### Data Supply for 2-D Sets in `.dat` Files

In a `.dat` file, members of a 2-D set are listed as space-separated pairs enclosed
in parentheses. The `param ... :=` block for a parameter indexed over a 2-D set
lists `origin destination value` triples.

```ampl
data;

set ORIG := GARY CLEV PITT ;
set DEST := FRA DET LAN WIN STL FRE LAF ;

# Sparse 2-D set: only valid (origin, destination) pairs
param: LINKS: cost :=
    GARY  DET  14    GARY  LAN  11    GARY  STL  16    GARY  LAF   8
    CLEV  FRA  27    CLEV  DET   9    CLEV  LAN  12    CLEV  WIN   9
    CLEV  STL  26    CLEV  LAF  17
    PITT  FRA  24    PITT  WIN  13    PITT  STL  28    PITT  FRE  99 ;
```

The `param: LINKS: cost :=` block simultaneously defines the members of `LINKS`
and the values of `cost`. Alternatively, the set and parameter may be specified
separately:

```ampl
# Explicit set membership, then parameter values
set LINKS := (GARY,DET) (GARY,LAN) (GARY,STL) (GARY,LAF)
             (CLEV,FRA) (CLEV,DET) (CLEV,LAN) (CLEV,WIN) (CLEV,STL) (CLEV,LAF)
             (PITT,FRA) (PITT,WIN) (PITT,STL) (PITT,FRE) ;

param cost :=
    GARY DET 14  GARY LAN 11  GARY STL 16  GARY LAF  8
    CLEV FRA 27  CLEV DET  9  CLEV LAN 12  CLEV WIN  9
    CLEV STL 26  CLEV LAF 17
    PITT FRA 24  PITT WIN 13  PITT STL 28  PITT FRE 99 ;
```

#### Example 2 — Sparse Subset: Transportation Model with Selected Links

When shipments are possible only on certain origin-destination pairs, the set `LINKS`
is a **sparse subset** of the full Cartesian product. The key difference from Example 1
is in the Supply and Demand constraints: because `Trans[i,j]` is defined only for
`(i,j) in LINKS`, both constraints must use the slice form `{(i,j) in LINKS}`
rather than iterating freely over `DEST` or `ORIG`.

In [ ]:
# Sparse transportation model (transp3.mod pattern — Fourer, Gay, Kernighan §6.2)
set ORIG;                                    # origin cities
set DEST;                                    # destination cities
set LINKS within ORIG cross DEST;            # sparse subset: only valid links

param supply {ORIG} >= 0;
param demand {DEST} >= 0;
param cost   {LINKS} >= 0;                   # cost only for declared links

check: sum {i in ORIG} supply[i] = sum {j in DEST} demand[j];

var Trans {LINKS} >= 0;                      # shipment only on declared links

minimize Total_Cost:
    sum {(i,j) in LINKS} cost[i,j] * Trans[i,j];

# Slice form: {(i,j) in LINKS} with i already defined selects all j such that (i,j) in LINKS
subject to Supply {i in ORIG}:
    sum {(i,j) in LINKS} Trans[i,j] = supply[i];

# Slice form: {(i,j) in LINKS} with j already defined selects all i such that (i,j) in LINKS
subject to Demand {j in DEST}:
    sum {(i,j) in LINKS} Trans[i,j] = demand[j];

#### Indexing over 2-D Sets: Slicing Semantics

The indexing expression `{(i,j) in LINKS}` changes meaning depending on context:

| Context | Indices already defined | What `{(i,j) in LINKS}` means |
|---|---|---|
| Objective or standalone sum | Neither `i` nor `j` | All pairs in `LINKS`; introduces both `i` and `j` |
| `Supply {i in ORIG}:` | `i` is fixed | All `j` such that `(i,j) in LINKS` — a row slice |
| `Demand {j in DEST}:` | `j` is fixed | All `i` such that `(i,j) in LINKS` — a column slice |
| With literal: `{("GARY",j) in LINKS}` | First component is literal | All `j` such that `("GARY",j) in LINKS` |

This **slice notation** is the canonical idiom for writing constraints that sum over
one dimension of a sparse 2-D set. It is both more concise and more efficient than
the guarded form:

```ampl
# Equivalent but less efficient — forces AMPL to enumerate all j in DEST first
sum {j in DEST: (i,j) in LINKS} Trans[i,j]

# Preferred — AMPL can directly look up the row for i without scanning all DEST
sum {(i,j) in LINKS} Trans[i,j]
```

**Important:** when iterating over a 2-D set without any pre-defined component,
the dummy indices must be declared as a pair `(i,j)`. Writing `{k in LINKS}` is
an error because `k` would need to be bound to a pair, not a single object.

#### 2-D Parameter Declarations: `{LINKS}` vs. `{ORIG, DEST}`

A parameter indexed over a 2-D set may be declared in three equivalent ways:

```ampl
param cost {LINKS} >= 0;                    # over named sparse set
param cost {ORIG, DEST} >= 0;               # over full Cartesian product (anonymous)
param cost {i in ORIG, j in DEST} >= 0;     # same, with named dummies
```

The first form — indexing over a **named set** — is preferred when:
- The set is sparse (not all pairs are valid)
- Multiple model components share the same index domain
- The set name conveys domain meaning (`LINKS`, `ARCS`, `ROUTES`)

The second and third forms implicitly use the full Cartesian product; they are
appropriate when every pair is valid and no named set is needed elsewhere.

Referencing values is identical for all three forms: `cost["GARY", "DET"]`.

In [ ]:
# Three equivalent forms for a parameter indexed over all (orig, dest) pairs
set ORIG;  set DEST;
set LINKS = ORIG cross DEST;

param cost1 {LINKS} >= 0;                    # Form 1: named set
param cost2 {ORIG, DEST} >= 0;               # Form 2: anonymous comma-syntax
param cost3 {i in ORIG, j in DEST} >= 0;     # Form 3: named dummies in comma-syntax

# All three are subscripted identically:
# cost1["GARY","DET"]  cost2["GARY","DET"]  cost3["GARY","DET"]

# When the set is sparse, only Form 1 (with a named sparse set) restricts the domain:
set LINKS_SPARSE within ORIG cross DEST;
param cost_sparse {LINKS_SPARSE} >= 0;       # cost only for valid links

#### Example 3 — Network Flow Model with Capacity

This example shows a directed network flow model where cities play both origin and
destination roles. The set `LINKS` is a sparse subset of `CITIES cross CITIES`.
A node balance constraint uses two slice expressions — one for incoming and one
for outgoing arcs.

In [ ]:
# Network flow model with capacity constraints
set CITIES;
set LINKS within (CITIES cross CITIES);      # directed arcs; self-loops allowed by 'within'

param supply   {CITIES} >= 0;                # net supply (positive) or demand (negative via check)
param demand   {CITIES} >= 0;
param cost     {LINKS}  >= 0;                # cost per unit shipped
param capacity {LINKS}  >= 0;                # capacity on each arc

check: sum {i in CITIES} supply[i] = sum {j in CITIES} demand[j];

var Ship {(i,j) in LINKS} >= 0, <= capacity[i,j];

minimize Total_Cost:
    sum {(i,j) in LINKS} cost[i,j] * Ship[i,j];

# Node balance: inflow + supply = outflow + demand
# Two slices: all arcs INTO k, and all arcs OUT OF k
subject to Balance {k in CITIES}:
    supply[k] + sum {(i,k) in LINKS} Ship[i,k]
    = demand[k] + sum {(k,j) in LINKS} Ship[k,j];

In the `Balance` constraint, `k` is the currently fixed node (already bound by
`{k in CITIES}`). The expression `{(i,k) in LINKS}` is a column slice: all `i`
such that there is an arc from `i` to `k`. The expression `{(k,j) in LINKS}` is
a row slice: all `j` such that there is an arc from `k` to `j`.

**Sample `.dat` data for this model:**

```ampl
data;
set CITIES := PITT NE SE BOS EWR BWI ATL MCO ;

set LINKS := (PITT,NE) (PITT,SE)
             (NE,BOS)  (NE,EWR)  (NE,BWI)
             (SE,EWR)  (SE,BWI)  (SE,ATL)  (SE,MCO) ;

param supply := PITT 450  NE 0  SE 0  BOS 0  EWR 0  BWI 0  ATL 0  MCO 0 ;
param demand := PITT 0  NE 0  SE 0  BOS 90  EWR 120  BWI 120  ATL 70  MCO 50 ;
param cost :=
    PITT NE 2.5  PITT SE 3.5
    NE BOS 1.7   NE EWR 0.7   NE BWI 1.3
    SE EWR 1.3   SE BWI 0.8   SE ATL 1.4   SE MCO 2.1 ;
param capacity :=
    PITT NE 250  PITT SE 250
    NE BOS 100   NE EWR 150   NE BWI 150
    SE EWR 150   SE BWI 150   SE ATL 100   SE MCO 100 ;
```

#### The `setof` Operator: Constructing 2-D Sets

The `setof` operator constructs a new set by evaluating an expression for each member
of an indexing set and collecting the results (duplicates discarded). For 2-D sets,
the argument to `setof` is a pair expression:

```ampl
set NAME = setof {INDEXING_EXPR} (COMPONENT1_EXPR, COMPONENT2_EXPR);
```

This is the standard way to **project** a higher-dimensional set down to a lower
dimension, or to **derive** a 2-D set from a condition on parameters.

**Syntax:** `setof` is followed by an indexing expression (using `{...}`) and then
an argument expression. The argument may be a single value (for 1-D results) or a
parenthesized tuple (for multi-dimensional results). Duplicate argument values are
silently discarded — the result is always a set.

In [ ]:
# setof examples

# --- Projecting a 3-D set down to a 2-D set ---
# ROUTES contains triples (origin, destination, product)
set ROUTES dimen 3;

# Extract the set of (orig, dest) pairs that appear in ROUTES
set LINKS = setof {(i,j,p) in ROUTES} (i,j);

# Extract the set of products that appear in ROUTES (1-D projection)
set PROD = setof {(i,j,p) in ROUTES} p;

# --- Building a 2-D set from a parameter condition ---
set ORIG;  set DEST;
param demand {ORIG, DEST} >= 0;

# Only include pairs with positive demand
set ACTIVE_LINKS = setof {i in ORIG, j in DEST: demand[i,j] > 0} (i,j);

# --- Constructing a set of peer pairs from a 1-D set ---
# setof can also build 2-D sets from conditions on 1-D sets
set JOBS;
param duration {JOBS} >= 0;
param deadline  {JOBS} >= 0;

# Set of (job1, job2) pairs where job1 must precede job2 due to deadline order
set PRECEDE = setof {j1 in JOBS, j2 in JOBS: deadline[j1] < deadline[j2]} (j1,j2);

#### Membership Test for Pairs

The `in` operator tests whether a specific pair is a member of a 2-D set:

```ampl
(i,j) in LINKS        # 1 if the pair (i,j) is a member of LINKS, else 0
(i,j) not in LINKS    # 1 if the pair (i,j) is NOT a member of LINKS, else 0
```

Membership tests appear in:
- **Indexing conditions**: `{j in DEST: (i,j) in LINKS}` — iterate only when the pair exists
- **Check statements**: `check {i in ORIG, j in DEST}: cost[i,j] >= 0;`
- **Parameter/variable definitions**: conditional expressions involving pair membership
- **`display` and `print` commands**: to inspect specific pairs interactively

In [ ]:
# Membership test for ordered pairs
set ORIG;  set DEST;
set LINKS within ORIG cross DEST;

param cost {LINKS} >= 0;

# Membership test in an indexing condition (equivalent to the slice form but less efficient)
# Supply constraint using an explicit membership guard:
# subject to Supply {i in ORIG}:
#     sum {j in DEST: (i,j) in LINKS} Trans[i,j] = supply[i];

# Membership test in a check statement
check {(i,j) in LINKS}: cost[i,j] >= 0;   # always satisfied given cost's >= 0 attribute

# Membership test in a conditional expression (used to compute a default safely)
param cost_or_zero {i in ORIG, j in DEST} =
    if (i,j) in LINKS then cost[i,j] else 0;

# Displaying a membership test (interactive use)
display ("GARY","DET") in LINKS;    # prints 1 if this arc is defined, 0 otherwise

#### Projections: Extracting Components of a 2-D Set

A **projection** reduces a 2-D set to a 1-D set by extracting one component.
AMPL does not provide a dedicated projection operator; projections are expressed
using `setof`:

```ampl
# First-component projection (all origins that appear in any link)
set USED_ORIG = setof {(i,j) in LINKS} i;

# Second-component projection (all destinations that appear in any link)
set USED_DEST = setof {(i,j) in LINKS} j;
```

Projections are useful when the component sets `ORIG` and `DEST` have not been
declared separately, or when only the cities actually involved in some link are relevant.
The `setof` operator discards duplicates, so each city appears at most once in the
projection.

In [ ]:
# Projections from a 2-D set
set ORIG;  set DEST;
set LINKS within ORIG cross DEST;

# Project onto first component: all origins that appear in at least one link
set ACTIVE_ORIG = setof {(i,j) in LINKS} i;

# Project onto second component: all destinations reachable from some origin
set ACTIVE_DEST = setof {(i,j) in LINKS} j;

# Derive projection from a 3-D set
set ROUTES dimen 3;   # triples (orig, dest, product)
set LINK_PAIRS  = setof {(i,j,p) in ROUTES} (i,j);   # 2-D projection
set ALL_ORIGINS = setof {(i,j,p) in ROUTES} i;        # 1-D projection: all origins used
set ALL_DESTS   = setof {(i,j,p) in ROUTES} j;        # 1-D projection: all destinations used
set ALL_PRODS   = setof {(i,j,p) in ROUTES} p;        # 1-D projection: all products shipped

# Projection with a condition: destinations reachable from a specific origin
param HOME symbolic;                                   # e.g., HOME := "GARY"
set FROM_HOME = setof {(i,j) in LINKS: i = HOME} j;  # all j reachable from HOME

#### Filtered 2-D Sets Using Conditions

A 2-D set may be built from a condition that references parameters. The resulting
set is **derived** in the AMPL sense: it is recomputed whenever the data it depends
on changes. Two common patterns are:

1. **Filter the full product**: keep only pairs satisfying a condition.
2. **Pair within the same set with a condition**: pairs of different elements from
   one set (e.g., for modeling transfers or comparisons).

In [ ]:
# Filtered 2-D sets

# Pattern 1: filter the full Cartesian product by a parameter condition
set DEST;  set PROD;
param demand {DEST, PROD} >= 0;

# Only (destination, product) pairs with positive demand
set DEMAND = {j in DEST, p in PROD: demand[j,p] > 0};

# Pattern 2: pairs of DISTINCT elements from one set (transfer arcs)
set ORIG;
set TRANSF = {i1 in ORIG, i2 in ORIG: i1 != i2};   # all pairs of distinct origins

# Pattern 3: ordered pairs from an ordered set (triangular structure — avoids duplicates)
set ORIG2 ordered;
set TRANSF_UPPER = {i1 in ORIG2, i2 in ORIG2: ord(i1) < ord(i2)};  # upper triangle only

# Pattern 4: setof with a filter condition (equivalent to Pattern 1 but as a setof)
set ACTIVE_DEMAND = setof {j in DEST, p in PROD: demand[j,p] > 0} (j,p);

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Declare sparse 2-D sets with <code>within ORIG cross DEST</code> rather than
<code>dimen 2</code> alone. The <code>within</code> attribute causes AMPL to validate
each data-supplied pair against the component sets immediately at data-load time,
catching errors such as a misspelled city name before they silently produce wrong
results at solve time.
</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Name the 2-D set when multiple model components share the same domain.
Writing <code>set LINKS within ORIG cross DEST;</code> and then indexing both
<code>param cost {LINKS}</code> and <code>var Trans {LINKS}</code> ensures that
cost and Trans are always indexed over exactly the same set of pairs, with a
single source of truth. Anonymous forms like <code>{ORIG, DEST}</code> in each
declaration are harder to keep consistent when the set of valid pairs changes.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
When iterating over a sparse 2-D set, dummy indices must be declared as a pair:
<code>{(i,j) in LINKS}</code>. Writing <code>{k in LINKS}</code> is a model error
because <code>k</code> would need to hold a pair, but AMPL dummy indices are scalar.
Similarly, writing <code>Trans[k]</code> where <code>k</code> was bound to a pair
would require two subscripts.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
With a sparse <code>LINKS</code> set, the supply constraint cannot use
<code>sum {j in DEST} Trans[i,j]</code> because <code>Trans[i,j]</code> is
undefined for pairs <code>(i,j)</code> not in <code>LINKS</code>. AMPL will
report: <em>"invalid subscript Trans['GARY','FRA']"</em> at model-generation time.
Use the slice form <code>sum {(i,j) in LINKS} Trans[i,j]</code> instead.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Sets of dimension two or higher cannot be declared <code>ordered</code> or
<code>circular</code> in AMPL — this is a language restriction. Consequently,
position-dependent functions such as <code>first</code>, <code>last</code>,
<code>next</code>, <code>prev</code>, and <code>ord</code> cannot be applied to
2-D sets. Attempting to add <code>ordered</code> to a 2-D set declaration is a
syntax error.
</div>

#### amplpy Equivalents

In amplpy, a 2-D set is assigned from a **list of tuples** or a **pandas DataFrame**
with a two-column MultiIndex.

| Task | amplpy pattern |
|---|---|
| Assign 2-D set from list of tuples | `ampl.set["LINKS"] = [("GARY","DET"), ("GARY","LAN"), ...]` |
| Assign 2-D set using `set_values` | `ampl.get_set("LINKS").set_values([("GARY","DET"), ...])` |
| Assign 2-D set from dict keys | `ampl.set["LINKS"] = list(cost_dict.keys())` |
| Assign from DataFrame with MultiIndex | `ampl.get_set("LINKS").set_values(df)` |
| Read 2-D set members back to Python | `ampl.set["LINKS"].members()` — returns list of tuples |
| Read as amplpy DataFrame | `ampl.set["LINKS"].get_values()` |
| Read as pandas DataFrame | `ampl.set["LINKS"].get_values().to_pandas()` |
| Membership test for a specific pair | `ampl.get_value('("GARY","DET") in LINKS')` |
| Assign set + indexed param together | `ampl.set_data(df, "LINKS")` where `df` has a 2-col index |

In [ ]:
import pandas as pd
from amplpy import AMPL

ampl = AMPL()

ampl.eval("""
    set ORIG;  set DEST;
    set LINKS within ORIG cross DEST;
    param cost {LINKS} >= 0;
""")

# Assign component sets
ampl.set["ORIG"] = ["GARY", "CLEV", "PITT"]
ampl.set["DEST"] = ["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"]

# --- Pattern 1: list of tuples (most direct) ---
cost_dict = {
    ("GARY", "DET"): 14, ("GARY", "LAN"): 11, ("GARY", "STL"): 16, ("GARY", "LAF"):  8,
    ("CLEV", "FRA"): 27, ("CLEV", "DET"):  9, ("CLEV", "LAN"): 12, ("CLEV", "WIN"):  9,
    ("CLEV", "STL"): 26, ("CLEV", "LAF"): 17,
    ("PITT", "FRA"): 24, ("PITT", "WIN"): 13, ("PITT", "STL"): 28, ("PITT", "FRE"): 99,
}

# LINKS from the dictionary keys (each key is already a (orig, dest) tuple)
ampl.set["LINKS"] = list(cost_dict.keys())
ampl.param["cost"] = cost_dict

# Read the set back as a list of tuples
links_members = ampl.set["LINKS"].members()   # e.g., [("GARY","DET"), ("GARY","LAN"), ...]
print("Number of links:", len(links_members))

# --- Pattern 2: pandas DataFrame with MultiIndex ---
ampl2 = AMPL()
ampl2.eval("""
    set ORIG;  set DEST;
    set LINKS within ORIG cross DEST;
    param cost {LINKS} >= 0;
""")
ampl2.set["ORIG"] = ["GARY", "CLEV", "PITT"]
ampl2.set["DEST"] = ["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"]

cost_df = pd.DataFrame(
    list(cost_dict.items()),
    columns=pd.MultiIndex.from_tuples([("ORIG", ""), ("DEST", "")])
    if False else ["idx", "cost"]                     # use simple columns for clarity
)
# Simpler: build as a Series with a MultiIndex, then set_data
cost_series = pd.Series(cost_dict, name="cost")
cost_series.index = pd.MultiIndex.from_tuples(cost_series.index, names=["ORIG", "DEST"])
cost_df2 = cost_series.reset_index()

# set_data initialises LINKS from the (ORIG, DEST) index columns and sets cost
ampl2.set_data(cost_df2.set_index(["ORIG", "DEST"]), "LINKS")

#### Edge Cases

**Edge case 1: An empty 2-D set is valid; indexing over it generates no instances.**

A 2-D set may have zero members. Constraints, sums, or parameters indexed over an
empty 2-D set produce zero instances — this is not an error. The `card` of the set
is 0 and `{} within LINKS` is always true.

In [ ]:
# Edge case 1: empty 2-D set
set ORIG := GARY CLEV PITT ;
set DEST := FRA DET LAN ;
set LINKS within ORIG cross DEST;

param cost {LINKS} >= 0;
var Trans {LINKS} >= 0;

# If no data is supplied for LINKS, it is empty.
# card(LINKS) = 0; the objective and constraints generate no instances.
# This is valid AMPL — the model compiles without error, the LP has no variables.
minimize Total_Cost:
    sum {(i,j) in LINKS} cost[i,j] * Trans[i,j];   # zero-term sum = 0; valid

# Membership test on an empty set always returns 0:
display ("GARY","DET") in LINKS;   # = 0 when LINKS is empty

**Edge case 2: Self-loops — a city as both origin and destination.**

When the first and second component sets are the same (e.g., `CITIES cross CITIES`),
self-loops `(i, i)` are included unless explicitly excluded by a condition.
Most network models do not want self-loops; exclude them by adding `i1 != i2`.

In [ ]:
# Edge case 2: self-loops in a set from one base set
set CITIES;

# This INCLUDES self-loops ("PITT","PITT") etc.
set ALL_ARCS = CITIES cross CITIES;

# This EXCLUDES self-loops (most network models prefer this)
set ARCS_NO_SELF = {i in CITIES, j in CITIES: i != j};

# Equivalently, using setof:
set ARCS_NO_SELF2 = setof {i in CITIES, j in CITIES: i != j} (i,j);

# Declaring a sparse set from data — AMPL validates components against CITIES
set LINKS within CITIES cross CITIES;   # data may or may not include self-loops

**Edge case 3: Using `dimen 2` without a `within` clause — no component validation.**

When `LINKS` is declared with only `dimen 2`, AMPL accepts any pair from the data,
including pairs whose components are not in `ORIG` or `DEST`. This can silently
produce wrong models. The `within` clause should always be preferred.

In [ ]:
# Edge case 3: dimen 2 without within — no validation
set ORIG;  set DEST;
set LINKS_UNSAFE dimen 2;              # AMPL accepts any pair; no component check
set LINKS_SAFE within ORIG cross DEST; # AMPL checks each pair against ORIG and DEST

# With LINKS_UNSAFE: if the data contains ("GARY", "TYPO_CITY") where "TYPO_CITY"
# is not in DEST, AMPL will silently accept it for LINKS_UNSAFE.
# Later, when param cost {LINKS_UNSAFE} is subscripted, the typo may produce
# wrong results or an uninformative error.

# With LINKS_SAFE: AMPL immediately reports an error such as:
#   "set LINKS_SAFE: member ('GARY','TYPO_CITY') not within {ORIG, DEST}"
#   at data-load time, before any solve is attempted.

**Edge case 4: `setof` with a filter over an empty index set returns the empty set.**

If the indexing expression in `setof` ranges over an empty set (or the condition
eliminates all members), the result is `{}`. This is well-defined and does not
produce an error.

In [ ]:
# Edge case 4: setof over an empty set or unsatisfied condition
set ORIG;  set DEST;
param cost {ORIG, DEST} >= 0;

# If cost is everywhere 0, this set is empty:
set EXPENSIVE = setof {i in ORIG, j in DEST: cost[i,j] > 1000} (i,j);
# card(EXPENSIVE) = 0; no error. Constraints indexed over EXPENSIVE generate no instances.

# If ORIG is empty, setof over ORIG cross DEST is also empty:
# set ORIG := ;   (empty data)
# set LINKS_DERIVED = setof {i in ORIG, j in DEST} (i,j);  --> {}

**Edge case 5: `{(i,j) in LINKS}` as a constraint indexing expression — zero-constraint generation.**

A constraint indexed over a 2-D set generates one instance per member of that set.
If the set is empty, zero constraint instances are generated — valid AMPL.
If the set has members but the slice for a particular `i` or `j` is empty (no
links from that origin or to that destination), the sum is zero, and the constraint
still exists and must be satisfied.

In [ ]:
# Edge case 5: origin with no outgoing links
set ORIG := GARY CLEV PITT BALT ;   # BALT added but has no links
set DEST := FRA DET LAN WIN STL FRE LAF ;
set LINKS within ORIG cross DEST;

param supply {ORIG} >= 0;
var Trans {LINKS} >= 0;

# For BALT, the set {("BALT",j) in LINKS} is empty.
# The Supply constraint for BALT becomes:  0 = supply["BALT"]
# This is satisfiable only if supply["BALT"] = 0; otherwise the LP is infeasible.
subject to Supply {i in ORIG}:
    sum {(i,j) in LINKS} Trans[i,j] = supply[i];

#### Cross-references

- [1.1.4. Set Operations](<1_1_4 set_operations.ipynb>) — introduction to `cross`, `within`, and sparse
  subsets; the present section provides the full treatment
- [1.1.5. Unordered Sets](<1_1_5 unordered_sets.ipynb>) — ordering restrictions on sets; 2-D sets
  cannot be declared `ordered` or `circular`
- [1.1.2. Indexing Expressions](<1_1_2 Indexing expressions.ipynb>) — the general grammar of indexing
  expressions including the `{(i,j) in S}` form and slice semantics
- [1.1.1. Set Declaration vs. Definition](<1_1_1 Set Declaration vs. Definition.ipynb>) — the `=`,
  `:=`, and `default` phrases; applies equally to 2-D sets